# Bridge Layer v1
## Per Bridge Layer Engineering Spec v3

Transforms the upstream forecast pipeline's reduced daily PV scenario package into
representative-day tables for the annual sizing MILP.

**Input**: `scenarios_joint_pv_load_reduced_K.parquet` + `load_deterministic_hourly.parquet` + `forecast_ghi_quantiles_daily.parquet`

**Output**: `repdays_metadata.parquet`, `calendar_map.parquet`, `scenarios_repdays_pv_reduced.parquet`, `risk_day_tags.parquet`, `bridge_report.json`, `bridge_run_metadata.json`

**Steps**: B0 → B1 → B2 → B3 → B4 → B5 → B6 → B7 → B8

In [1]:
import pandas as pd
import numpy as np
import json
import hashlib
from pathlib import Path
from datetime import datetime
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

try:
    from sklearn_extra.cluster import KMedoids
except ImportError:
    from sklearn.cluster import KMeans
    class KMedoids:
        """Fallback k-medoids using KMeans initialization + PAM swap."""
        def __init__(self, n_clusters=5, random_state=42, metric='euclidean', method='alternate'):
            self.n_clusters = n_clusters
            self.random_state = random_state
            self.metric = metric
        def fit_predict(self, X):
            km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
            labels = km.fit_predict(X)
            # Find actual medoids (closest point to centroid in each cluster)
            self.medoid_indices_ = []
            for c in range(self.n_clusters):
                mask = labels == c
                if not mask.any():
                    self.medoid_indices_.append(0)
                    continue
                cluster_pts = np.where(mask)[0]
                centroid = X[mask].mean(axis=0)
                dists = np.linalg.norm(X[cluster_pts] - centroid, axis=1)
                self.medoid_indices_.append(cluster_pts[np.argmin(dists)])
            self.medoid_indices_ = np.array(self.medoid_indices_)
            return labels
    print('Using built-in KMedoids fallback')

print('Imports OK')

Using built-in KMedoids fallback
Imports OK


In [2]:
# ============================================================
# BRIDGE CONFIGURATION
# ============================================================
BCFG = dict(
    # --- Paths (relative to notebook location) ---
    pipeline_dir   = '../pipeline_outputs',
    forecast_file  = '../pipeline_outputs/forecast_ghi_quantiles_daily.parquet',
    load_file      = '../pipeline_outputs/load_deterministic_hourly.parquet',
    reduced_file   = '../pipeline_outputs/scenarios_joint_pv_load_reduced_5.parquet',
    output_dir     = '../bridge_outputs_v1',

    # --- Case year (spec §0.2) ---
    case_year_start = '2024-11-01',  # ground-truth load start
    case_year_end   = '2025-10-31',  # ground-truth load end

    # --- Scenario generation (reuse forecast pipeline settings) ---
    N_scenarios = 500,
    K_reduced   = 5,
    random_seed = 42,

    # --- PV conversion params (same as forecast pipeline) ---
    site_lat  = 25.0377,
    site_lon  = 121.5149,
    tz        = 'Asia/Taipei',
    pv_dc_rated_kw = 50.0,
    gamma_pdc      = -0.0047,
    inv_eff        = 0.96,
    dc_ac_ratio    = 1.2,
    inoct          = 49.0,

    # --- Risk-day parameters (spec §5.1, §9) ---
    risk_pct       = 0.10,   # top 10% by stress score
    peak_window    = (9, 17),  # 09:00–17:00 for peak-hour descriptors

    # --- Body clustering (spec §5.2, §9) ---
    n_body_repdays = 20,     # target number of body clusters
    cluster_metric = 'euclidean',
    stratify_by    = 'month',  # stratify clustering by month

    # --- Quantiles for descriptors ---
    quantiles = [round(0.05 + 0.05 * i, 2) for i in range(19)],
)

BCFG['pv_ac_rated_kw'] = BCFG['pv_dc_rated_kw'] / BCFG['dc_ac_ratio']
Path(BCFG['output_dir']).mkdir(parents=True, exist_ok=True)
np.random.seed(BCFG['random_seed'])

print(f"Case year: {BCFG['case_year_start']} – {BCFG['case_year_end']}")
print(f"Risk-day %: {BCFG['risk_pct']*100:.0f}%")
print(f"Body repday target: {BCFG['n_body_repdays']}")

Case year: 2024-11-01 – 2025-10-31
Risk-day %: 10%
Body repday target: 20


---
## Pre-Step: Generate Full-Year Scenarios

The forecast pipeline only produced 7 demo days of reduced scenarios.
The bridge needs all 365 days. We reuse the forecast pipeline functions to generate
full-year GHI scenarios → GHI→PV → k-medoids reduction for every test day.

In [3]:
from scipy import stats
from scipy.interpolate import interp1d
import pvlib
from pvlib import location

# ── Load upstream artifacts ──
forecast_cqr = pd.read_parquet(BCFG['forecast_file'])
load_det = pd.read_parquet(BCFG['load_file'])

# Ensure tz-naive timestamps for consistency
for col in ['target_day_local', 'target_time_local']:
    if hasattr(forecast_cqr[col].dtype, 'tz') and forecast_cqr[col].dt.tz is not None:
        forecast_cqr[col] = forecast_cqr[col].dt.tz_localize(None)
    if hasattr(load_det[col].dtype, 'tz') and load_det[col].dt.tz is not None:
        load_det[col] = load_det[col].dt.tz_localize(None)

# Filter to case year test set
fc_test = forecast_cqr[forecast_cqr['split_name'] == 'test'].copy()
fc_test['target_day_local'] = pd.to_datetime(fc_test['target_day_local'])
case_mask = (fc_test['target_day_local'] >= BCFG['case_year_start']) & \
            (fc_test['target_day_local'] <= BCFG['case_year_end'])
fc_case = fc_test[case_mask]

target_days = sorted(fc_case['target_day_local'].unique())
print(f"Forecast days in case year: {len(target_days)}")
print(f"Range: {target_days[0]} – {target_days[-1]}")
print(f"Load profile rows: {len(load_det)}")

Forecast days in case year: 365
Range: 2024-11-01 00:00:00 – 2025-10-31 00:00:00
Load profile rows: 8760


In [4]:
def quantile_function(quantiles, q_values):
    """Piecewise linear Q(u) from 19Q."""
    taus = np.array([0.0] + list(quantiles) + [1.0])
    vals = np.concatenate([[q_values[0]], q_values, [q_values[-1]]])
    return interp1d(taus, vals, kind='linear', bounds_error=False,
                    fill_value=(vals[0], vals[-1]))


def generate_full_year_scenarios(fc, load_det, target_days, N, K):
    """Generate N GHI scenarios → GHI→PV → reduce to K per day for all target days."""
    quantiles = BCFG['quantiles']
    q_cols = [f'q{q:.2f}' for q in quantiles]

    # Load lookup (tz-naive)
    load_lookup = load_det[['target_time_local', 'load_kw']].drop_duplicates('target_time_local')
    hourly_mean = load_det.groupby('hour')['load_kw'].mean()

    # PV conversion setup
    site = location.Location(BCFG['site_lat'], BCFG['site_lon'], tz=BCFG['tz'])
    monthly_temp = {1:16,2:17,3:19,4:23,5:26,6:28,7:30,8:30,9:28,10:25,11:21,12:18}

    all_reduced = []

    for day in tqdm(target_days, desc='Full-year scenarios'):
        grp = fc[fc['target_day_local'] == day].sort_values('horizon_hour').head(24)
        if len(grp) < 24:
            continue

        # ── GHI scenario generation (24-dim Gaussian copula on GHI hours) ──
        ghi_qfuncs = []
        for _, row in grp.iterrows():
            ghi_qfuncs.append(quantile_function(quantiles, row[q_cols].values.astype(float)))

        Sigma = np.eye(24)
        for i in range(24):
            for j in range(24):
                Sigma[i, j] = np.exp(-0.3 * abs(i - j))

        rng = np.random.default_rng(BCFG['random_seed'] + hash(str(day)) % 10000)
        z = rng.multivariate_normal(np.zeros(24), Sigma, size=N)
        u = stats.norm.cdf(z)

        horizon_hours = grp['horizon_hour'].values
        issue_day = day - pd.Timedelta(days=1)

        rows = []
        for s in range(N):
            for h in range(24):
                tt = pd.Timestamp(day) + pd.Timedelta(hours=h)
                ghi = max(float(ghi_qfuncs[h](u[s, h])), 0)

                # Load lookup
                lk = load_lookup[load_lookup['target_time_local'] == tt]
                load_kw = float(lk['load_kw'].iloc[0]) if len(lk) > 0 else float(hourly_mean.get(h, 0))

                # ── GHI→PV inline ──
                poa = max(ghi, 0)
                month = tt.month
                t_air = monthly_temp.get(month, 25)
                t_cell = t_air + poa / 800.0 * (BCFG['inoct'] - 20)
                p_dc = BCFG['pv_dc_rated_kw'] * (poa / 1000.0) * (1 + BCFG['gamma_pdc'] * (t_cell - 25))
                p_dc = max(p_dc, 0)
                pv_kw = min(BCFG['pv_ac_rated_kw'], p_dc * BCFG['inv_eff'])
                pv_kw = max(pv_kw, 0)

                rows.append({
                    'target_day_local': day,
                    'target_time_local': tt,
                    'issue_day_local': issue_day,
                    'horizon_hour': horizon_hours[h],
                    'scenario_id': s,
                    'pv_available_kw': pv_kw,
                    'load_kw': load_kw,
                    'ghi_wm2': ghi,
                })

        sc = pd.DataFrame(rows)

        # ── Nighttime PV = 0 ──
        times_local = pd.DatetimeIndex(sc['target_time_local']).tz_localize(BCFG['tz'])
        solpos = site.get_solarposition(times_local)
        sc.loc[solpos['elevation'].values <= 0, 'pv_available_kw'] = 0.0

        # ── k-medoids reduction on PV trajectories (24-dim) ──
        pivoted = sc.pivot_table(index='scenario_id', columns=sc['target_time_local'].dt.hour,
                                 values='pv_available_kw', aggfunc='first').fillna(0)
        pivoted.columns = [f'pv_h{h}' for h in pivoted.columns]

        scaler = StandardScaler()
        X = scaler.fit_transform(pivoted.values)
        actual_K = min(K, len(pivoted))
        kmed = KMedoids(n_clusters=actual_K, random_state=BCFG['random_seed'],
                        metric='euclidean', method='alternate')
        labels = kmed.fit_predict(X)

        medoid_sids = pivoted.index[kmed.medoid_indices_].tolist()
        cluster_sizes = pd.Series(labels).value_counts().sort_index()
        probs = (cluster_sizes / cluster_sizes.sum()).values

        for k_idx, (sid, prob) in enumerate(zip(medoid_sids, probs)):
            med = sc[sc['scenario_id'] == sid].copy()
            med['scenario_id'] = k_idx
            med['probability_pi'] = prob
            all_reduced.append(med)

    result = pd.concat(all_reduced, ignore_index=True)
    print(f"Full-year reduced scenarios: {result.shape}")
    print(f"Days: {result['target_day_local'].nunique()}, K={K} per day")
    return result


# Check if full-year file already exists (skip regeneration)
fullyear_path = Path(BCFG['output_dir']) / f"scenarios_fullyear_reduced_{BCFG['K_reduced']}.parquet"
if fullyear_path.exists():
    print(f"Loading cached full-year scenarios from {fullyear_path}")
    scenarios_full = pd.read_parquet(fullyear_path)
else:
    scenarios_full = generate_full_year_scenarios(
        fc_case, load_det, target_days,
        N=BCFG['N_scenarios'], K=BCFG['K_reduced']
    )
    scenarios_full.to_parquet(fullyear_path, index=False)
    print(f"Saved to {fullyear_path}")

print(f"\nDays: {scenarios_full['target_day_local'].nunique()}")
print(f"Shape: {scenarios_full.shape}")

Loading cached full-year scenarios from ../bridge_outputs_v1/scenarios_fullyear_reduced_5.parquet



Days: 365
Shape: (43800, 9)


---
## B0: Define Analysis Year and Available Target Days

Fix the case year, retain only target days within it.

In [5]:
# ── B0: Define analysis year ──
scenarios_full['target_day_local'] = pd.to_datetime(scenarios_full['target_day_local'])
scenarios_full['target_time_local'] = pd.to_datetime(scenarios_full['target_time_local'])

case_start = pd.Timestamp(BCFG['case_year_start'])
case_end = pd.Timestamp(BCFG['case_year_end'])

sc = scenarios_full[
    (scenarios_full['target_day_local'] >= case_start) &
    (scenarios_full['target_day_local'] <= case_end)
].copy()

calendar_days = sorted(sc['target_day_local'].unique())
n_days = len(calendar_days)
K = sc.groupby('target_day_local')['scenario_id'].nunique().iloc[0]

print(f"B0: {n_days} calendar days in case year")
print(f"    {case_start.date()} – {case_end.date()}")
print(f"    K = {K} scenarios per day")

B0: 365 calendar days in case year
    2024-11-01 – 2025-10-31
    K = 5 scenarios per day


---
## B1: Reorganize Daily Scenario Package into Day Bundles

Merge K scenarios with 24h load into a single daily object per calendar day.

In [6]:
# ── B1: Build day bundles ──
# For each day, create a structured bundle with PV scenarios and load

day_bundles = {}
for day, grp in sc.groupby('target_day_local'):
    grp = grp.sort_values(['scenario_id', 'target_time_local'])

    # PV matrix: (K, 24)
    pv_matrix = grp.pivot_table(index='scenario_id', columns=grp['target_time_local'].dt.hour,
                                values='pv_available_kw', aggfunc='first').fillna(0)

    # Load vector: (24,) — deterministic, same across scenarios
    load_vec = grp[grp['scenario_id'] == 0].sort_values('target_time_local')['load_kw'].values
    if len(load_vec) < 24:
        load_vec = np.zeros(24)

    # Scenario probabilities
    probs = grp.drop_duplicates('scenario_id').set_index('scenario_id')['probability_pi']

    day_bundles[day] = {
        'pv_matrix': pv_matrix.values,       # (K, 24)
        'load_24h': load_vec[:24],            # (24,)
        'probability_pi': probs.values,       # (K,)
        'scenario_ids': pv_matrix.index.tolist(),
        'date': day,
        'month': day.month,
        'dayofweek': day.dayofweek,
        'is_weekday': int(day.dayofweek < 5),
    }

print(f"B1: {len(day_bundles)} day bundles created")
sample_day = list(day_bundles.keys())[0]
b = day_bundles[sample_day]
print(f"    Sample: {sample_day.date()}, PV shape={b['pv_matrix'].shape}, load shape={b['load_24h'].shape}")

B1: 365 day bundles created
    Sample: 2024-11-01, PV shape=(5, 24), load shape=(24,)


---
## B2: Build Day Descriptors

Per spec §4: compute day-level features for representative-day selection and risk scoring.

- **Load descriptors** (§4.1): daily total, peak, peak-window average, workday/weekend, month/season
- **PV scenario descriptors** (§4.2): P10/P50/P90 daily PV energy, P10 afternoon shortage, scenario-weighted ramp
- **Net-load stress** (§4.3): `Stress_i = PeakLoad_i - PV_P10_peakWindow_i`

In [7]:
# ── B2: Build day descriptors ──

def get_season(month):
    if month in [3, 4, 5]: return 'spring'
    if month in [6, 7, 8]: return 'summer'
    if month in [9, 10, 11]: return 'autumn'
    return 'winter'

peak_start, peak_end = BCFG['peak_window']
afternoon_start, afternoon_end = 13, 17  # for PV shortage indicator

descriptors = []
for day, b in day_bundles.items():
    pv = b['pv_matrix']      # (K, 24)
    load = b['load_24h']     # (24,)
    pi = b['probability_pi'] # (K,)

    # ── Load descriptors (deterministic) ──
    daily_load_total = load.sum()
    daily_peak_load = load.max()
    peak_hours = load[peak_start:peak_end]
    avg_load_peak = peak_hours.mean() if len(peak_hours) > 0 else 0

    # ── PV scenario descriptors (aggregated from K scenarios) ──
    daily_pv_per_scenario = pv.sum(axis=1)  # (K,)
    pv_p10 = np.percentile(daily_pv_per_scenario, 10)
    pv_p50 = np.percentile(daily_pv_per_scenario, 50)
    pv_p90 = np.percentile(daily_pv_per_scenario, 90)
    pv_weighted_mean = np.average(daily_pv_per_scenario, weights=pi)

    # P10 afternoon shortage (hours 13-17)
    afternoon_pv = pv[:, afternoon_start:afternoon_end]  # (K, 4)
    afternoon_totals = afternoon_pv.sum(axis=1)
    pv_p10_afternoon = np.percentile(afternoon_totals, 10)

    # Scenario-weighted ramp indicator (max hourly drop in PV across scenarios)
    hourly_diffs = np.diff(pv, axis=1)  # (K, 23)
    max_drops = hourly_diffs.min(axis=1)  # worst drop per scenario
    ramp_indicator = np.average(max_drops, weights=pi)

    # Worst-period low-PV tail: average PV in bottom 10% scenarios during peak
    peak_pv = pv[:, peak_start:peak_end]
    peak_pv_totals = peak_pv.sum(axis=1)
    tail_threshold = np.percentile(peak_pv_totals, 10)
    tail_mask = peak_pv_totals <= tail_threshold
    worst_tail_pv = peak_pv_totals[tail_mask].mean() if tail_mask.any() else peak_pv_totals.min()

    # ── Net-load stress (spec §4.3) ──
    # PV P10 during peak window
    pv_p10_peak_window = np.percentile(peak_pv_totals, 10) if len(peak_pv_totals) > 0 else 0
    stress = daily_peak_load - pv_p10_peak_window / max(peak_end - peak_start, 1)

    descriptors.append({
        'calendar_day': day,
        'month': b['month'],
        'season': get_season(b['month']),
        'dayofweek': b['dayofweek'],
        'is_weekday': b['is_weekday'],
        # Load
        'daily_load_total': daily_load_total,
        'daily_peak_load': daily_peak_load,
        'avg_load_peak_window': avg_load_peak,
        # PV
        'pv_p10_daily': pv_p10,
        'pv_p50_daily': pv_p50,
        'pv_p90_daily': pv_p90,
        'pv_weighted_mean': pv_weighted_mean,
        'pv_p10_afternoon': pv_p10_afternoon,
        'ramp_indicator': ramp_indicator,
        'worst_tail_pv': worst_tail_pv,
        # Stress
        'stress_score': stress,
    })

desc_df = pd.DataFrame(descriptors)
desc_df = desc_df.sort_values('calendar_day').reset_index(drop=True)

print(f"B2: {len(desc_df)} day descriptors")
print(f"\nStress score stats:")
print(desc_df['stress_score'].describe())
print(f"\nPV P50 daily stats:")
print(desc_df['pv_p50_daily'].describe())

B2: 365 day descriptors

Stress score stats:
count     365.000000
mean     3170.956065
std       984.083520
min      1063.921444
25%      2409.614975
50%      3025.292130
75%      3995.458635
max      5151.216355
Name: stress_score, dtype: float64

PV P50 daily stats:
count    365.000000
mean     157.406942
std       70.261542
min       27.788843
25%       88.061719
50%      164.799814
75%      217.392538
max      298.271651
Name: pv_p50_daily, dtype: float64


---
## B3: Risk-Day Tagging

Per spec §5.1: tag days that must not be averaged away.
- Top X% by net-load stress
- Top X% by daily peak load
- Top X% by PV P10 shortage (low afternoon PV)
- Monthly coverage check: if a month has no risk day, supplement with highest-stress day in that month

In [8]:
# ── B3: Risk-day tagging ──
risk_pct = BCFG['risk_pct']
n_risk_candidates = max(1, int(np.ceil(n_days * risk_pct)))

# Rank by multiple criteria
desc_df['rank_stress'] = desc_df['stress_score'].rank(ascending=False)
desc_df['rank_peak_load'] = desc_df['daily_peak_load'].rank(ascending=False)
desc_df['rank_low_pv'] = desc_df['pv_p10_afternoon'].rank(ascending=True)  # low = risky

# Tag if in top X% for ANY criterion
risk_threshold = n_risk_candidates
desc_df['risk_stress'] = desc_df['rank_stress'] <= risk_threshold
desc_df['risk_peak'] = desc_df['rank_peak_load'] <= risk_threshold
desc_df['risk_low_pv'] = desc_df['rank_low_pv'] <= risk_threshold

# Union of all risk tags
desc_df['is_risk_day'] = desc_df['risk_stress'] | desc_df['risk_peak'] | desc_df['risk_low_pv']

# Build tag_reason
def get_tag_reason(row):
    reasons = []
    if row['risk_stress']: reasons.append('high_stress')
    if row['risk_peak']: reasons.append('peak_load')
    if row['risk_low_pv']: reasons.append('low_pv_afternoon')
    return '; '.join(reasons) if reasons else ''

desc_df['tag_reason'] = desc_df.apply(get_tag_reason, axis=1)

# Monthly coverage check: ensure every month has at least 1 risk day
months_with_risk = desc_df[desc_df['is_risk_day']].groupby('month')['calendar_day'].count()
all_months = desc_df['month'].unique()
supplemented = []

for m in all_months:
    if m not in months_with_risk.index or months_with_risk[m] == 0:
        # Supplement: highest stress day in this month
        month_days = desc_df[desc_df['month'] == m]
        best_idx = month_days['stress_score'].idxmax()
        desc_df.loc[best_idx, 'is_risk_day'] = True
        desc_df.loc[best_idx, 'tag_reason'] = 'monthly_supplement'
        supplemented.append((m, desc_df.loc[best_idx, 'calendar_day']))

risk_days = desc_df[desc_df['is_risk_day']]['calendar_day'].tolist()
body_days = desc_df[~desc_df['is_risk_day']]['calendar_day'].tolist()

print(f"B3: {len(risk_days)} risk days tagged ({len(risk_days)/n_days*100:.1f}%)")
print(f"    {len(body_days)} body days remaining")
print(f"    Supplemented months: {supplemented if supplemented else 'none'}")
print(f"\nRisk days by month:")
print(desc_df[desc_df['is_risk_day']].groupby('month')['calendar_day'].count())

B3: 75 risk days tagged (20.5%)
    290 body days remaining
    Supplemented months: [(np.int64(8), Timestamp('2025-08-29 00:00:00'))]

Risk days by month:
month
1      3
2      2
3      2
4      4
5      4
6      2
7      2
8      1
9     19
10    19
11     8
12     9
Name: calendar_day, dtype: int64


---
## B4: Cluster the Remaining Body Days

Per spec §5.2: k-medoids on day descriptors (NOT raw 24×K matrix).
Stratify first by month to avoid unreasonable cross-season clustering.
Allocate cluster count proportionally to each month's body-day count.

In [9]:
# ── B4: Body-day clustering ──
body_desc = desc_df[~desc_df['is_risk_day']].copy()
n_body_target = BCFG['n_body_repdays']

# Descriptor features for clustering
feature_cols = ['daily_load_total', 'daily_peak_load', 'avg_load_peak_window',
                'pv_p10_daily', 'pv_p50_daily', 'pv_p90_daily',
                'pv_weighted_mean', 'pv_p10_afternoon', 'ramp_indicator',
                'stress_score', 'is_weekday']

# Stratify by month: allocate clusters proportionally
month_counts = body_desc.groupby('month').size()
total_body = len(body_desc)

# Proportional allocation (at least 1 per month)
month_clusters = {}
remaining = n_body_target
for m in month_counts.index:
    alloc = max(1, int(round(month_counts[m] / total_body * n_body_target)))
    month_clusters[m] = alloc
    remaining -= alloc

# Adjust if over/under allocated
while sum(month_clusters.values()) > n_body_target:
    # Reduce from largest
    m_max = max(month_clusters, key=month_clusters.get)
    if month_clusters[m_max] > 1:
        month_clusters[m_max] -= 1
while sum(month_clusters.values()) < n_body_target:
    # Add to largest month
    m_max = max(month_counts.index, key=lambda m: month_counts[m])
    month_clusters[m_max] += 1

print(f"Body-day cluster allocation by month:")
for m in sorted(month_clusters.keys()):
    print(f"  Month {m:2d}: {month_counts.get(m, 0):3d} body days → {month_clusters[m]} clusters")

# Run k-medoids within each month stratum
body_desc = body_desc.copy()
body_desc['cluster_id'] = -1
body_desc['is_medoid'] = False

global_cluster_id = 0
scaler = StandardScaler()

for m, n_clust in month_clusters.items():
    month_mask = body_desc['month'] == m
    month_data = body_desc[month_mask]

    if len(month_data) == 0:
        continue

    n_clust = min(n_clust, len(month_data))

    X = month_data[feature_cols].fillna(0).values
    X_scaled = scaler.fit_transform(X)

    kmed = KMedoids(n_clusters=n_clust, random_state=BCFG['random_seed'],
                    metric=BCFG['cluster_metric'], method='alternate')
    labels = kmed.fit_predict(X_scaled)

    for local_c in range(n_clust):
        cluster_mask = labels == local_c
        cluster_indices = month_data.index[cluster_mask]
        body_desc.loc[cluster_indices, 'cluster_id'] = global_cluster_id

        # Mark medoid
        medoid_idx = month_data.index[kmed.medoid_indices_[local_c]]
        body_desc.loc[medoid_idx, 'is_medoid'] = True

        global_cluster_id += 1

n_actual_clusters = body_desc['cluster_id'].nunique()
n_medoids = body_desc['is_medoid'].sum()
print(f"\nB4: {n_actual_clusters} body clusters, {n_medoids} medoids selected")

Body-day cluster allocation by month:
  Month  1:  28 body days → 1 clusters
  Month  2:  26 body days → 1 clusters
  Month  3:  29 body days → 2 clusters
  Month  4:  26 body days → 2 clusters
  Month  5:  27 body days → 2 clusters
  Month  6:  28 body days → 2 clusters
  Month  7:  29 body days → 2 clusters
  Month  8:  30 body days → 2 clusters
  Month  9:  11 body days → 1 clusters
  Month 10:  12 body days → 1 clusters
  Month 11:  22 body days → 2 clusters
  Month 12:  22 body days → 2 clusters

B4: 20 body clusters, 20 medoids selected


---
## B5–B6: Select Medoid Days, Build Weights and Calendar Map

- **B5**: Each body cluster's medoid is a repday; each risk day is a standalone repday.
  Medoid repdays directly inherit original K PV scenarios (no re-synthesis).
- **B6**: Each calendar day maps to one `repday_id`; cluster size = repday weight.

In [10]:
# ── B5 + B6: Build repdays_metadata and calendar_map ──

repdays = []
calendar_map_rows = []
repday_id_counter = 0

# ── Body medoid repdays ──
medoid_days = body_desc[body_desc['is_medoid']]

for _, med_row in medoid_days.iterrows():
    cluster_id = med_row['cluster_id']
    source_date = med_row['calendar_day']

    # Weight = cluster size
    cluster_members = body_desc[body_desc['cluster_id'] == cluster_id]
    weight = len(cluster_members)

    rid = f"body_{repday_id_counter:03d}"
    repdays.append({
        'repday_id': rid,
        'source_date': source_date,
        'repday_type': 'body',
        'weight': weight,
        'month_tag': med_row['month'],
        'season_tag': med_row['season'],
        'cluster_id': int(cluster_id),
    })

    # Map all cluster members to this repday
    for _, member in cluster_members.iterrows():
        calendar_map_rows.append({
            'calendar_day': member['calendar_day'],
            'repday_id': rid,
            'month_id': member['month'],
            'day_type': 'workday' if member['is_weekday'] else 'weekend',
            'is_risk_day': False,
        })

    repday_id_counter += 1

# ── Risk repdays (each is standalone, weight = 1) ──
risk_desc = desc_df[desc_df['is_risk_day']]

for _, risk_row in risk_desc.iterrows():
    source_date = risk_row['calendar_day']
    rid = f"risk_{repday_id_counter:03d}"

    repdays.append({
        'repday_id': rid,
        'source_date': source_date,
        'repday_type': 'risk',
        'weight': 1,
        'month_tag': risk_row['month'],
        'season_tag': risk_row['season'],
        'cluster_id': -1,
    })

    calendar_map_rows.append({
        'calendar_day': source_date,
        'repday_id': rid,
        'month_id': risk_row['month'],
        'day_type': 'workday' if risk_row['is_weekday'] else 'weekend',
        'is_risk_day': True,
    })

    repday_id_counter += 1

repdays_meta = pd.DataFrame(repdays)
calendar_map = pd.DataFrame(calendar_map_rows)

# ── Verify ──
total_weight = repdays_meta['weight'].sum()
coverage = len(calendar_map)
n_repdays = len(repdays_meta)
n_body_rep = len(repdays_meta[repdays_meta['repday_type'] == 'body'])
n_risk_rep = len(repdays_meta[repdays_meta['repday_type'] == 'risk'])

print(f"B5-B6: {n_repdays} total repdays ({n_body_rep} body + {n_risk_rep} risk)")
print(f"    Total weight: {total_weight} (should be {n_days})")
print(f"    Calendar map coverage: {coverage}/{n_days} days")
print(f"    Weight check: {'PASS' if total_weight == n_days else 'FAIL'}")
print(f"    Coverage check: {'PASS' if coverage == n_days else 'FAIL'}")

B5-B6: 95 total repdays (20 body + 75 risk)
    Total weight: 365 (should be 365)
    Calendar map coverage: 365/365 days
    Weight check: PASS
    Coverage check: PASS


---
## B7: Assemble Repday-Level Scenario Tables

Per spec §6.3: for each repday, bring in the original K reduced PV scenarios
from its `source_date`. No scenario re-generation — bridge only selects, maps, and reorganizes.

Output schema: `repday_id, scenario_id, probability_pi, hour, pv_available_kw, load_kw, source_date`

In [11]:
# ── B7: Assemble repday scenario tables ──

repday_scenarios = []

for _, rep in repdays_meta.iterrows():
    rid = rep['repday_id']
    source = rep['source_date']

    # Get original K scenarios for this source date
    day_sc = sc[sc['target_day_local'] == source].copy()

    if len(day_sc) == 0:
        print(f"WARNING: No scenarios for repday {rid} source_date {source}")
        continue

    for _, row in day_sc.iterrows():
        repday_scenarios.append({
            'repday_id': rid,
            'scenario_id': row['scenario_id'],
            'probability_pi': row['probability_pi'],
            'hour': row['target_time_local'].hour + 1,  # 1..24 per spec
            'pv_available_kw': row['pv_available_kw'],
            'load_kw': row['load_kw'],
            'source_date': source,
        })

scenarios_repdays = pd.DataFrame(repday_scenarios)

# Verify probability sums
pi_check = scenarios_repdays.groupby('repday_id').apply(
    lambda g: g.drop_duplicates('scenario_id')['probability_pi'].sum()
)

print(f"B7: scenarios_repdays_pv_reduced shape: {scenarios_repdays.shape}")
print(f"    Unique repdays: {scenarios_repdays['repday_id'].nunique()}")
print(f"    Pi sum range: {pi_check.min():.4f} – {pi_check.max():.4f} (should be ~1.0)")
print(f"    Hours per repday-scenario: {scenarios_repdays.groupby(['repday_id', 'scenario_id']).size().unique()}")

B7: scenarios_repdays_pv_reduced shape: (11400, 7)
    Unique repdays: 95
    Pi sum range: 1.0000 – 1.0000 (should be ~1.0)
    Hours per repday-scenario: [24]


---
## B8: QA, Reports, and Output Bridge-Ready Artifacts

Per spec §8: check coverage, weights, risk-day coverage, scenario completeness.
Write all 6 output artifacts.

In [12]:
# ── B8: QA checks ──

qa_checks = {}

# 1. Calendar map coverage = 100%
mapped_days = set(calendar_map['calendar_day'])
all_days = set(calendar_days)
coverage_pct = len(mapped_days & all_days) / len(all_days) * 100
qa_checks['calendar_map_coverage'] = f"{coverage_pct:.1f}%"
qa_checks['calendar_map_coverage_pass'] = coverage_pct == 100.0

# 2. Sum of weights = number of calendar days
weight_sum = repdays_meta['weight'].sum()
qa_checks['weight_sum'] = int(weight_sum)
qa_checks['weight_sum_target'] = n_days
qa_checks['weight_sum_pass'] = weight_sum == n_days

# 3. Scenario probabilities sum to 1 per repday
pi_sums = scenarios_repdays.groupby('repday_id').apply(
    lambda g: g.drop_duplicates('scenario_id')['probability_pi'].sum())
qa_checks['pi_sum_range'] = f"{pi_sums.min():.4f} – {pi_sums.max():.4f}"
qa_checks['pi_sum_pass'] = bool((pi_sums > 0.999).all() and (pi_sums < 1.001).all())

# 4. Traceability: all repday source_dates exist in scenarios
repday_sources = set(repdays_meta['source_date'])
scenario_days = set(scenarios_repdays['source_date'])
qa_checks['source_date_traceable'] = repday_sources == scenario_days

# 5. Risk-day coverage per month
risk_by_month = desc_df[desc_df['is_risk_day']].groupby('month').size()
qa_checks['risk_days_per_month'] = risk_by_month.to_dict()
qa_checks['all_months_have_risk'] = len(risk_by_month) == len(desc_df['month'].unique())

# Print QA summary
print("=" * 55)
print("  B8: QA CHECKS")
print("=" * 55)
for k, v in qa_checks.items():
    status = ""
    if isinstance(v, bool):
        status = " ✓" if v else " ✗ FAIL"
    print(f"  {k}: {v}{status}")

all_pass = all(v for k, v in qa_checks.items() if k.endswith('_pass'))
print(f"\n  Overall: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")

  B8: QA CHECKS
  calendar_map_coverage: 100.0%
  calendar_map_coverage_pass: True ✓
  weight_sum: 365
  weight_sum_target: 365
  weight_sum_pass: True
  pi_sum_range: 1.0000 – 1.0000
  pi_sum_pass: True ✓
  source_date_traceable: True ✓
  risk_days_per_month: {1: 3, 2: 2, 3: 2, 4: 4, 5: 4, 6: 2, 7: 2, 8: 1, 9: 19, 10: 19, 11: 8, 12: 9}
  all_months_have_risk: True ✓

  Overall: ALL CHECKS PASSED


In [13]:
# ── B8: Write all output artifacts ──
out = Path(BCFG['output_dir'])

# 1. repdays_metadata.parquet
repdays_meta.to_parquet(out / 'repdays_metadata.parquet', index=False)

# 2. calendar_map.parquet
calendar_map.to_parquet(out / 'calendar_map.parquet', index=False)

# 3. scenarios_repdays_pv_reduced.parquet
scenarios_repdays.to_parquet(out / 'scenarios_repdays_pv_reduced.parquet', index=False)

# 4. risk_day_tags.parquet
risk_tags = desc_df[['calendar_day', 'stress_score', 'tag_reason', 'is_risk_day',
                      'daily_peak_load', 'pv_p10_afternoon', 'pv_p10_daily',
                      'month', 'season']].copy()
risk_tags = risk_tags.rename(columns={'is_risk_day': 'selected_flag', 'stress_score': 'risk_score'})
risk_tags.to_parquet(out / 'risk_day_tags.parquet', index=False)

# 5. bridge_report.json
bridge_report = {
    'case_year': f"{BCFG['case_year_start']} – {BCFG['case_year_end']}",
    'n_calendar_days': n_days,
    'K_scenarios_per_day': int(K),
    'n_repdays_total': n_repdays,
    'n_body_repdays': n_body_rep,
    'n_risk_repdays': n_risk_rep,
    'risk_pct_threshold': BCFG['risk_pct'],
    'risk_day_count': len(risk_days),
    'risk_day_dates': [str(d.date()) for d in risk_days],
    'risk_day_criteria': ['high_stress (top 10%)', 'peak_load (top 10%)',
                          'low_pv_afternoon (top 10%)', 'monthly_supplement'],
    'body_cluster_count': n_actual_clusters,
    'body_cluster_stratification': BCFG['stratify_by'],
    'cluster_metric': BCFG['cluster_metric'],
    'qa_checks': {k: str(v) for k, v in qa_checks.items()},
}

with open(out / 'bridge_report.json', 'w') as f:
    json.dump(bridge_report, f, indent=2, default=str)

# 6. bridge_run_metadata.json
run_meta = {
    'run_id': hashlib.md5(str(datetime.now()).encode()).hexdigest()[:12],
    'created_at': datetime.now().isoformat(),
    'config_hash': hashlib.md5(json.dumps(BCFG, default=str).encode()).hexdigest()[:12],
    'random_seed': BCFG['random_seed'],
    'version': 'bridge_v1',
    'feature_set': feature_cols,
    'distance_metric': BCFG['cluster_metric'],
    'n_body_clusters': n_actual_clusters,
    'risk_pct': BCFG['risk_pct'],
    'N_raw_scenarios': BCFG['N_scenarios'],
    'K_reduced_scenarios': BCFG['K_reduced'],
}

with open(out / 'bridge_run_metadata.json', 'w') as f:
    json.dump(run_meta, f, indent=2, default=str)

# Summary
print("Bridge outputs written:")
for f in sorted(out.glob('*')):
    size = f.stat().st_size
    print(f"  {f.name:45s} {size/1024:8.1f} KB")

Bridge outputs written:
  bridge_report.json                                 2.2 KB
  bridge_run_metadata.json                           0.5 KB
  calendar_map.parquet                               6.9 KB
  repdays_metadata.parquet                           6.1 KB
  risk_day_tags.parquet                             20.5 KB
  scenarios_fullyear_reduced_5.parquet             539.2 KB
  scenarios_repdays_pv_reduced.parquet              71.2 KB


---
## Summary

| Artifact | Spec Reference | Downstream User |
|----------|---------------|-----------------|
| `repdays_metadata.parquet` | §7.1 | Annual sizing MILP |
| `calendar_map.parquet` | §7.2 | Annual sizing MILP (SOC linkage, monthly max-demand proxy) |
| `scenarios_repdays_pv_reduced.parquet` | §7.3 | Annual sizing MILP |
| `risk_day_tags.parquet` | §2 | Bridge QA / thesis analysis |
| `bridge_report.json` | §8 | Engineering and thesis appendix |
| `bridge_run_metadata.json` | §8 | Reproducibility |